# Go1 PACT/DACT and ABL3 reconstruction evaluation

Both frozen models are scored on each recorded history, applied torque and pre-reset target. Tables and figures keep rollout source, condition and terrain separate. Valid samples from failed episodes are retained.

The old notebook selected a single task prefix and used obsolete privileged-state slices plus a repeated success mask. This notebook discovers **both methods through manifests** and uses the standalone analysis functions. Old CSVs receive an explicit unavailable report because their target layout and timing cannot be established reliably.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reconstruction_eval").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from reconstruction_eval.analysis import analyze, legacy_availability
from reconstruction_eval.reporting import discover_datasets, dataset_inventory, comparison_table, plot_metric


## Select a comparison

Set `DATASETS` to collection directories containing the same two evaluated checkpoints. Leave it empty to discover collections under `DATA_ROOT`; inspect the inventory before running analysis. Discovery does not filter on `pact` or `abl` filenames. Use one experiment directory to avoid mixing checkpoints or duplicate replay collections.

A controlled comparison requires matching completed scenario sets from both rollout sources. `CONTROLLED = False` permits a single source but does not establish balanced trajectory coverage. Physics and probes are optional; probes require at least five independent scenario families.


In [ ]:
DATA_ROOT = ROOT / "eval"
DATASETS = []  # Example: [DATA_ROOT / "pact_nominal", DATA_ROOT / "abl3_nominal"]
OUTPUT = DATA_ROOT / "analysis"
LEGACY_ROOT = ROOT / "legged_gym/scripts/exp_data_corl_10/recon_01"
CONTROLLED = True
CONTACT_THRESHOLD_N = 1.0
TRANSITION_WINDOW_S = 0.03
RECOVERY_WINDOW_S = 0.75
MOMENTUM_WINDOWS_S = [0.05, 0.1]
DYNAMICS_NORMALIZATION = [100.0, 10.0, 10.0]  # N, N m, N m
BOOTSTRAP_EPISODES = 1000
ANALYSIS_SEED = 1
RUN_DYNAMICS = False  # Requires Pinocchio; uses recorded states, no simulation.
RUN_PROBES = False    # Frozen latents only; identical grouped splits for both models.


In [ ]:
selected_datasets = [Path(p) for p in DATASETS] if DATASETS else discover_datasets(DATA_ROOT)
inventory = dataset_inventory(selected_datasets)
display(inventory)
legacy_files = sorted(LEGACY_ROOT.rglob("go1_*_decoder_eval.csv"))
legacy_report = legacy_availability(legacy_files)
if not legacy_report.empty:
    display(legacy_report)
if not selected_datasets:
    print("No versioned collections selected. Set DATASETS or DATA_ROOT above; legacy availability is shown separately.")


## Compute and export

The CLI runs the same function:

```bash
conda run -n genesis_lr python -m reconstruction_eval.analysis eval/pact_nominal eval/abl3_nominal --output eval/analysis --controlled --bootstrap 1000 --dynamics --plots
```

Add `--probes` for the frozen-latent ridge comparison. CSVs have explicit identifier columns and no implicit DataFrame index. RMSE is calculated from squared-error sums. Confidence intervals resample episodes conditional on these checkpoints; they do **not** measure training-seed uncertainty.


In [ ]:
summary = pd.DataFrame()
if selected_datasets:
    summary = analyze(
        selected_datasets, OUTPUT, controlled=CONTROLLED,
        contact_threshold=CONTACT_THRESHOLD_N,
        transition_window=TRANSITION_WINDOW_S,
        recovery_window=RECOVERY_WINDOW_S,
        momentum_windows=MOMENTUM_WINDOWS_S,
        normalization=DYNAMICS_NORMALIZATION,
        dynamics=RUN_DYNAMICS, probes=RUN_PROBES,
        bootstrap=BOOTSTRAP_EPISODES, seed=ANALYSIS_SEED,
    )
    print(f"Saved analysis to {OUTPUT}")


## Reconstruction on common histories

Payload errors are in kg, and CoM offsets in metres (axis scores and pooled components). The primary next-step motion score selects **projected gravity, angular velocity, joint position and joint velocity**, in each checkpoint's normalized units. Command and previous-action bookkeeping are separate. If checkpoint scales differ, normalized errors retain those checkpoint-specific scales.

The general decoder excludes GRFs; force predictions come from the separately trained torque-conditioned head. Explicit current-step estimates and next-step errors are separately named: the runners at `9c4e6a1` actually train explicit outputs against post-step labels despite their storage comments.


In [ ]:
PRIMARY = ["payload", "com", "com_x", "com_y", "com_z", "motion_observation", "grf"]
display(comparison_table(summary, PRIMARY))
BOOKKEEPING = ["command", "previous_action", "explicit_current_payload", "explicit_next_payload",
               "explicit_current_com", "explicit_next_com", "explicit_current_linear_velocity", "explicit_next_linear_velocity"]
display(comparison_table(summary, BOOKKEEPING))
for metric in ["payload", "com", "motion_observation", "grf"]:
    if not summary.empty:
        fig = plot_metric(summary, metric)
        if fig is not None:
            fig.savefig(OUTPUT / f"{metric}_rmse.png", dpi=160, bbox_inches="tight")
            plt.show()
            plt.close(fig)


## Contact phases and disturbance recovery

Phase labels use measured force norms and chronological samples within each environment/episode. Touchdown and liftoff extend forward for the selected transition duration. Gaps have unknown phase. Targets are not smoothed, and predicted forces do not determine phases.

Select a phase and event window below. Per-foot/component scores remain available in `summary.csv`. An instantaneous injection row is `impulse_event`; the configured post-event interval is `recovery`. Momentum windows containing an injection are reported separately too.


In [ ]:
PHASE = "touchdown"  # steady_stance, swing, touchdown, liftoff, unknown, all
WINDOW = "all"       # ordinary, impulse_event, recovery, all
if not summary.empty:
    force_metrics = sorted(m for m in summary.metric.unique() if m == "grf" or m.startswith("grf_"))
    display(comparison_table(summary, force_metrics, phase=PHASE, window=WINDOW))
    display(comparison_table(summary, PRIMARY, window="impulse_event"))
    display(comparison_table(summary, PRIMARY, window="recovery"))


## Whole-body dynamics and momentum

Enable `RUN_DYNAMICS` above to compute these offline with the common Pinocchio implementation for both methods. Residuals use recorded simulator inertias, actual payload/CoM, applied torque and the final physics-substep acceleration interval. Floating-base translation, rotation and joint residuals are separate; the normalized aggregate uses the declared fixed scales.

Momentum uses whole-body centroidal momentum, gravity, contact wrenches and measured injection jumps. Angular balance uses a fixed origin at each window's starting whole-body CoM. Predicted and measured endpoint forces use identical right-endpoint quadrature; scene-step contact impulses provide a finer reference. Net link forces are applied at recorded link origins, a point-force approximation without contact couples. Measured-force residuals quantify numerical/model consistency.


In [ ]:
if RUN_DYNAMICS and not summary.empty:
    physics_metrics = sorted(m for m in summary.metric.unique() if "dynamics_" in m or "momentum_" in m)
    display(comparison_table(summary, physics_metrics))
    display(comparison_table(summary, physics_metrics, window="recovery"))
else:
    print("Dynamics/momentum disabled. Set RUN_DYNAMICS = True to calculate them from the recorded data.")


## Frozen probes, on-policy scores and exclusions

Ridge probes use deterministic frozen latents only, identical scenario-grouped train/validation/test splits, train-only normalization and the same regularization grid. Shared scenario families stay in the same split across sources, conditions and terrains. Held-out errors include a constant train-mean reference.

The separate on-policy table below describes each policy's own distribution; use the common-history tables for representation comparison. Exclusion reasons can overlap. Check valid episode/sample counts before interpreting any score.


In [ ]:
if RUN_PROBES and not summary.empty:
    display(pd.read_csv(OUTPUT / "probes.csv"))
display(comparison_table(summary, PRIMARY, comparison="on_policy"))
if not summary.empty:
    display(pd.read_csv(OUTPUT / "exclusions.csv"))
    display(pd.read_csv(OUTPUT / "unavailable.csv"))
    print("Protocol: analysis_manifest.json; method/collection details: inventory.csv and input manifest.json files.")
